# Chapter 89: AI for Cloud & Endpoint Security

**Volume 4 — Security Operations**

**The breach didn't come from a zero-day.** It came from an S3 bucket misconfigured
two years ago, an IAM role with wildcard permissions nobody audited, and a container
running as root that your scanner never reached. The attack surface grew faster than
the team. AI doesn't replace security engineers — it gives them **machine-speed
perception over a surface area that is otherwise too large to see clearly.**

### What You Will Build — 5 Cloud & Endpoint Security Engines

| Demo | Technique | What It Catches |
|------|-----------|----------------|
| **1. Cloud Config Drift Detector** | Baseline diff + risk scoring | IAM/S3/SG misconfigurations vs known-good |
| **2. Endpoint Behavioral Baseline** | Process execution pattern modeling | Unusual child processes, persistence mechanisms |
| **3. Container Security Scanner** | Dockerfile best-practice checker | Privilege escalation risks in container configs |
| **4. Cloud Workload Anomaly Detector** | Statistical z-score on CPU/network/API metrics | Cryptomining, exfiltration, C2 callback patterns |
| **5. Full Cloud Security Posture Pipeline** | Drift + risk + LLM remediation | End-to-end CSPM with actionable fix plans |

### The Core Insight
> **Misconfiguration — not zero-days — is the #1 cause of cloud breaches.
> AI adds anomaly detection on top of rule-based CSPM: it flags what rule writers
> never anticipated, correlates resource relationships for true risk scoring,
> and generates remediation plans that developers can actually act on.**
>
> *Think of your cloud as an IS-IS topology. Each resource is a router, each IAM
> permission is a link. Attack path analysis is computing attacker paths before
> they do — the same way you compute ECMP paths before traffic arrives.*

In [ ]:
# pip install anthropic
import os, json, re, math, time, random
import statistics
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from collections import defaultdict

try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')

api_key = os.environ.get('ANTHROPIC_API_KEY', '')
USE_API = bool(api_key)

if USE_API:
    from anthropic import Anthropic
    client = Anthropic()
    print("LLM analyst: ACTIVE (Anthropic API)")
else:
    print("LLM analyst: SIMULATION MODE (set ANTHROPIC_API_KEY for real analysis)")

def llm_analyze(prompt: str, max_tokens: int = 200) -> str:
    if USE_API:
        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}]
        )
        return resp.content[0].text.strip()
    return "Simulation: " + prompt[:80] + "..."

print("Setup complete.")

## Demo 1: Cloud Config Drift Detector

**Configuration drift** is when your live cloud environment diverges from the
known-good baseline — a developer changes an IAM policy without a change ticket,
someone opens a security group port for "testing" and forgets to close it,
an S3 bucket ACL gets modified during an incident and never reverted.

This demo compares live cloud config snapshots against a known-good baseline
across three resource types:

| Resource Type | High-Risk Drift Patterns |
|---------------|-------------------------|
| **IAM Policies** | Wildcard `*` actions, overly broad resource scope, new admin grants |
| **Security Groups** | 0.0.0.0/0 ingress rules, port 22/3389 to internet, new permissive rules |
| **S3 Bucket ACLs** | Public read/write, no encryption, no logging, cross-account access |

Each drift finding is risk-scored by severity (0.0-1.0). The scores drive
prioritization — a CSPM that generates 10,000 findings without priority
is just as useless as no CSPM at all.

*Network analogy: This is exactly like comparing a router's running-config
to its startup-config after an incident. Any delta is a drift event.
The difference: cloud configs change 1000x faster and across thousands of resources.*

In [ ]:
# ── Demo 1: Cloud Config Drift Detector ───────────────────────────────────────

@dataclass
class DriftFinding:
    """A single configuration drift finding."""
    resource_id: str
    resource_type: str      # 'iam_policy', 'security_group', 's3_bucket'
    field: str              # which config field changed
    baseline_value: str     # what the known-good value was
    current_value: str      # what it is now
    risk_score: float       # 0.0 = no risk, 1.0 = critical
    severity: str           # CRITICAL / HIGH / MEDIUM / LOW
    description: str

# ── Risk scoring rules per drift type ─────────────────────────────────────────

def score_iam_drift(resource_id: str, baseline: dict, current: dict) -> List[DriftFinding]:
    """Diff IAM policy baseline vs current. Return scored findings."""
    findings = []

    # Check: wildcard actions added
    b_actions = set(baseline.get("allowed_actions", []))
    c_actions = set(current.get("allowed_actions", []))
    new_actions = c_actions - b_actions
    for action in new_actions:
        if "*" in action:
            findings.append(DriftFinding(
                resource_id=resource_id, resource_type="iam_policy",
                field="allowed_actions",
                baseline_value="(not present)", current_value=action,
                risk_score=0.90, severity="CRITICAL",
                description=f"Wildcard action '{action}' added — grants overly broad permissions"
            ))
        else:
            findings.append(DriftFinding(
                resource_id=resource_id, resource_type="iam_policy",
                field="allowed_actions",
                baseline_value="(not present)", current_value=action,
                risk_score=0.40, severity="MEDIUM",
                description=f"New action '{action}' added to policy"
            ))

    # Check: resource scope widened
    b_resources = set(baseline.get("resource_scope", []))
    c_resources = set(current.get("resource_scope", []))
    if "*" in c_resources and "*" not in b_resources:
        findings.append(DriftFinding(
            resource_id=resource_id, resource_type="iam_policy",
            field="resource_scope",
            baseline_value=str(b_resources), current_value="*",
            risk_score=0.85, severity="CRITICAL",
            description="Resource scope changed to '*' — applies to ALL resources in account"
        ))

    # Check: MFA condition removed
    if baseline.get("require_mfa") and not current.get("require_mfa"):
        findings.append(DriftFinding(
            resource_id=resource_id, resource_type="iam_policy",
            field="require_mfa",
            baseline_value="true", current_value="false",
            risk_score=0.75, severity="HIGH",
            description="MFA condition removed from IAM policy — allows access without MFA"
        ))

    return findings

def score_sg_drift(resource_id: str, baseline: dict, current: dict) -> List[DriftFinding]:
    """Diff security group baseline vs current ingress rules."""
    findings = []
    b_rules = {r["port"]: r for r in baseline.get("ingress_rules", [])}
    c_rules = {r["port"]: r for r in current.get("ingress_rules", [])}

    for port, rule in c_rules.items():
        cidr = rule.get("cidr", "")
        if port not in b_rules:
            # New rule added
            if cidr in ("0.0.0.0/0", "::/0"):
                risk = 0.95 if port in (22, 3389, 5432, 3306, 27017) else 0.70
                sev = "CRITICAL" if risk >= 0.90 else "HIGH"
                findings.append(DriftFinding(
                    resource_id=resource_id, resource_type="security_group",
                    field=f"ingress_port_{port}",
                    baseline_value="(rule not present)", current_value=f"port {port} from {cidr}",
                    risk_score=risk, severity=sev,
                    description=f"Port {port} opened to internet ({cidr}) — potential attack surface"
                ))
            else:
                findings.append(DriftFinding(
                    resource_id=resource_id, resource_type="security_group",
                    field=f"ingress_port_{port}",
                    baseline_value="(rule not present)", current_value=f"port {port} from {cidr}",
                    risk_score=0.30, severity="LOW",
                    description=f"New ingress rule: port {port} from {cidr}"
                ))
        elif c_rules[port].get("cidr") != b_rules[port].get("cidr"):
            # Existing rule widened
            if cidr in ("0.0.0.0/0", "::/0"):
                findings.append(DriftFinding(
                    resource_id=resource_id, resource_type="security_group",
                    field=f"ingress_port_{port}_cidr",
                    baseline_value=b_rules[port].get("cidr"), current_value=cidr,
                    risk_score=0.85, severity="CRITICAL",
                    description=f"Port {port} CIDR widened to internet — baseline was more restrictive"
                ))

    return findings

def score_s3_drift(resource_id: str, baseline: dict, current: dict) -> List[DriftFinding]:
    """Diff S3 bucket configuration baseline vs current."""
    findings = []

    if baseline.get("public_access_blocked") and not current.get("public_access_blocked"):
        findings.append(DriftFinding(
            resource_id=resource_id, resource_type="s3_bucket",
            field="public_access_blocked",
            baseline_value="true", current_value="false",
            risk_score=0.95, severity="CRITICAL",
            description="S3 public access block REMOVED — bucket may now be publicly accessible"
        ))

    if baseline.get("encryption_enabled") and not current.get("encryption_enabled"):
        findings.append(DriftFinding(
            resource_id=resource_id, resource_type="s3_bucket",
            field="encryption_enabled",
            baseline_value="true", current_value="false",
            risk_score=0.70, severity="HIGH",
            description="S3 encryption disabled — data at rest no longer encrypted"
        ))

    if baseline.get("logging_enabled") and not current.get("logging_enabled"):
        findings.append(DriftFinding(
            resource_id=resource_id, resource_type="s3_bucket",
            field="logging_enabled",
            baseline_value="true", current_value="false",
            risk_score=0.45, severity="MEDIUM",
            description="S3 access logging disabled — forensic trail lost"
        ))

    new_cross_account = set(current.get("cross_account_principals", [])) - \
                        set(baseline.get("cross_account_principals", []))
    for principal in new_cross_account:
        findings.append(DriftFinding(
            resource_id=resource_id, resource_type="s3_bucket",
            field="cross_account_principals",
            baseline_value="(not present)", current_value=principal,
            risk_score=0.80, severity="HIGH",
            description=f"New cross-account access granted to: {principal}"
        ))

    return findings

# ── Simulate baseline vs current configs ──────────────────────────────────────
print("=" * 68)
print("CLOUD CONFIG DRIFT DETECTOR — CSPM FINDING REPORT")
print("=" * 68)

all_findings: List[DriftFinding] = []

# IAM policy drift
all_findings += score_iam_drift(
    "prod-lambda-execution-role",
    baseline={"allowed_actions": ["s3:GetObject", "logs:CreateLogGroup"],
              "resource_scope": ["arn:aws:s3:::prod-data-bucket/*"],
              "require_mfa": True},
    current={"allowed_actions": ["s3:GetObject", "logs:CreateLogGroup", "s3:*", "iam:PassRole"],
             "resource_scope": ["*"],
             "require_mfa": False},
)

# Security group drift
all_findings += score_sg_drift(
    "sg-prod-rds-db-01",
    baseline={"ingress_rules": [
        {"port": 5432, "cidr": "10.0.1.0/24"},
    ]},
    current={"ingress_rules": [
        {"port": 5432, "cidr": "0.0.0.0/0"},   # widened to internet!
        {"port": 22,   "cidr": "0.0.0.0/0"},   # SSH opened to internet!
    ]},
)

# S3 drift
all_findings += score_s3_drift(
    "prod-customer-data-bucket",
    baseline={"public_access_blocked": True, "encryption_enabled": True,
              "logging_enabled": True, "cross_account_principals": []},
    current={"public_access_blocked": False, "encryption_enabled": True,
             "logging_enabled": False,
             "cross_account_principals": ["arn:aws:iam::999888777666:root"]},
)

# Sort and display
all_findings.sort(key=lambda f: f.risk_score, reverse=True)

print(f"{'#':<3} {'Severity':<10} {'Score':>6} {'Resource':<35} {'Field'}")
print("-" * 85)
for i, f in enumerate(all_findings, 1):
    print(f"{i:<3} {f.severity:<10} {f.risk_score:>6.2f} {f.resource_id:<35} {f.field}")
    print(f"     {f.description}")

# LLM remediation for top critical finding
critical = [f for f in all_findings if f.severity == "CRITICAL"]
if critical:
    top = critical[0]
    print(f"\n[LLM Remediation — {top.resource_id}]")
    analysis = llm_analyze(
        f"Cloud security drift finding: {top.resource_id} ({top.resource_type}).\n"
        f"Issue: {top.description}\n"
        f"Risk score: {top.risk_score:.2f} ({top.severity}).\n"
        f"Provide: 1) specific fix (AWS CLI or Terraform), 2) blast radius, 3) rollback step. "
        f"Under 100 words.",
        max_tokens=150
    )
    print(analysis)

print(f"\n[Drift Detector] {len(all_findings)} findings | "
      f"CRITICAL={sum(1 for f in all_findings if f.severity=='CRITICAL')} | "
      f"HIGH={sum(1 for f in all_findings if f.severity=='HIGH')}")

## Demo 2: Endpoint Behavioral Baseline

**Signature-based AV is dead for zero-day detection.** Fileless malware has no file
to scan. Living-off-the-land (LotL) attacks use only built-in Windows tools. Process
injection hides in legitimate processes. The only reliable detection surface is
**what the process *does*, not what it looks like.**

This demo builds a **process execution behavioral baseline** — learning which parent
processes spawn which children, what command-line patterns are normal, and which
combinations are the signatures of known attack chains:

| Attack Chain | Pattern | MITRE Technique |
|-------------|---------|----------------|
| Macro malware | `winword.exe -> powershell.exe` | T1566 (Phishing) + T1059 |
| LSASS dump | `lsass.exe` as child or direct memory access | T1003.001 |
| C2 beaconing | `svchost.exe -> outbound_connection` to new IP | T1071 |
| Persistence | Any process spawning `schtasks.exe` or `reg.exe` | T1053 / T1547 |
| Lateral movement | `powershell.exe -> net.exe` or `wmic.exe` | T1021 |

The anomaly score combines: parent-child pair novelty + command-line entropy
+ known attack chain matching + unusual child count vs baseline.

In [ ]:
# ── Demo 2: Endpoint Behavioral Baseline ──────────────────────────────────────

@dataclass
class ProcessEvent:
    """A process creation event from EDR telemetry."""
    process_name: str
    parent_name: str
    command_line: str
    user_context: str        # SYSTEM, user, service
    network_connection: bool # did the process make outbound connections?
    child_count: int         # children spawned in this session
    host: str
    hour: int

# 90-day baseline: known-normal parent -> children pairs
NORMAL_PARENT_CHILD = {
    "explorer.exe":     {"chrome.exe", "firefox.exe", "outlook.exe", "teams.exe",
                         "notepad.exe", "calc.exe", "mspaint.exe"},
    "services.exe":     {"svchost.exe", "lsass.exe", "wininit.exe"},
    "svchost.exe":      {"taskhost.exe", "conhost.exe", "wermgr.exe"},
    "chrome.exe":       {"chrome.exe", "crashpad_handler.exe"},
    "outlook.exe":      {"splwow64.exe", "winword.exe", "excel.exe"},
    "powershell.exe":   {"conhost.exe"},                # PowerShell normally only spawns conhost
    "cmd.exe":          {"conhost.exe", "ipconfig.exe", "ping.exe", "tracert.exe"},
    "msiexec.exe":      {"msiexec.exe", "cmd.exe"},
}

# High-risk command line patterns (regex fragments)
SUSPICIOUS_CMDLINE_PATTERNS = [
    (r'(?i)-enc(?:odedcommand)?\s+[A-Za-z0-9+/]{20}', 0.80,  "Base64-encoded PowerShell command"),
    (r'(?i)invoke-expression|iex\s*\(',              0.75,  "PowerShell IEX (download + execute)"),
    (r'(?i)downloadstring|webclient',                0.70,  "In-memory download pattern"),
    (r'(?i)mimikatz|sekurlsa|lsadump',               0.95,  "Credential dumping tool keyword"),
    (r'(?i)\-nop.*\-w.*hidden',                     0.65,  "PowerShell hidden window, no profile"),
    (r'(?i)net user.*\/add|net localgroup.*administrators', 0.85, "Account creation / admin add"),
    (r'(?i)schtasks.*\/create',                     0.60,  "Scheduled task persistence"),
    (r'(?i)reg.*add.*run',                           0.55,  "Registry Run key persistence"),
]

# Known attack chain parent-child pairs (immediate flag)
ATTACK_CHAIN_PAIRS = {
    ("winword.exe",   "powershell.exe"): (0.90, "T1566+T1059: Macro -> PowerShell execution"),
    ("winword.exe",   "cmd.exe"):        (0.85, "T1566: Office macro spawning cmd shell"),
    ("excel.exe",     "powershell.exe"): (0.90, "T1566+T1059: Excel macro -> PowerShell"),
    ("powershell.exe","net.exe"):        (0.75, "T1021: PowerShell lateral movement via net"),
    ("powershell.exe","wmic.exe"):       (0.80, "T1047: WMI-based lateral movement"),
    ("cmd.exe",       "powershell.exe"): (0.55, "T1059: CMD spawning PowerShell (common in LotL)"),
    ("explorer.exe",  "powershell.exe"): (0.40, "Uncommon but seen in some legitimate tools"),
    ("lsass.exe",     "cmd.exe"):        (0.95, "T1003.001: LSASS spawning shell — critical"),
}

def analyze_process_event(event: ProcessEvent) -> dict:
    """
    Score a process event against the behavioral baseline.
    Returns anomaly score (0.0=normal, 1.0=critical) and finding list.
    """
    findings = []
    scores = []

    # Check 1: Known attack chain parent-child pair
    pair = (event.parent_name.lower(), event.process_name.lower())
    if pair in ATTACK_CHAIN_PAIRS:
        score, technique = ATTACK_CHAIN_PAIRS[pair]
        findings.append({"type": "ATTACK_CHAIN", "score": score, "detail": technique})
        scores.append(score)

    # Check 2: Unknown parent-child pair (not in baseline)
    parent_lower = event.parent_name.lower()
    child_lower  = event.process_name.lower()
    normal_children = NORMAL_PARENT_CHILD.get(parent_lower, set())
    if normal_children and child_lower not in normal_children and pair not in ATTACK_CHAIN_PAIRS:
        findings.append({
            "type": "UNKNOWN_CHILD",
            "score": 0.45,
            "detail": f"{event.parent_name} spawning {event.process_name} not in 90-day baseline"
        })
        scores.append(0.45)

    # Check 3: Suspicious command line patterns
    for pattern, pattern_score, description in SUSPICIOUS_CMDLINE_PATTERNS:
        if re.search(pattern, event.command_line):
            findings.append({"type": "CMDLINE_PATTERN", "score": pattern_score, "detail": description})
            scores.append(pattern_score)

    # Check 4: Unusual child count vs baseline
    if event.child_count > 8:
        child_score = min((event.child_count - 8) / 10.0, 0.50)
        findings.append({"type": "HIGH_CHILD_COUNT", "score": child_score,
                         "detail": f"{event.child_count} children spawned (baseline: <8)"})
        scores.append(child_score)

    # Check 5: SYSTEM-context process making unexpected network connections
    if event.network_connection and event.user_context == "SYSTEM" and \
       event.process_name.lower() not in {"svchost.exe", "lsass.exe", "wininit.exe"}:
        findings.append({"type": "SYSTEM_NETCONN", "score": 0.65,
                         "detail": f"SYSTEM-context {event.process_name} made outbound connection"})
        scores.append(0.65)

    final_score = max(scores) if scores else 0.0
    severity = "CRITICAL" if final_score >= 0.80 else ("HIGH" if final_score >= 0.60 else
               ("MEDIUM" if final_score >= 0.40 else "LOW"))

    return {"score": round(final_score, 3), "severity": severity,
            "findings": findings, "finding_count": len(findings)}

# ── Test process events ────────────────────────────────────────────────────────
print("=" * 68)
print("ENDPOINT BEHAVIORAL BASELINE — PROCESS EVENT ANALYSIS")
print("=" * 68)

test_events = [
    (ProcessEvent("chrome.exe",      "explorer.exe",  "chrome.exe --profile-directory=Default",
                  "jsmith",      False, 3,  "WKSTN-101", 10), "Normal browser launch"),
    (ProcessEvent("powershell.exe",  "winword.exe",
                  "powershell.exe -nop -w hidden -EncodedCommand JABjACAAPQAgACcAaQBp",
                  "jsmith",      True,  2,  "WKSTN-101", 14), "Macro -> PS + base64 payload"),
    (ProcessEvent("net.exe",         "powershell.exe",
                  "net user hacker P@ssw0rd123 /add && net localgroup administrators hacker /add",
                  "jsmith",      False, 1,  "WKSTN-101", 14), "Account persistence"),
    (ProcessEvent("schtasks.exe",    "cmd.exe",
                  "schtasks /create /tn 'SystemUpdate' /sc daily /tr 'powershell -enc ABCD'",
                  "SYSTEM",      False, 0,  "SRV-DEV-02", 3), "Scheduled task persistence"),
    (ProcessEvent("conhost.exe",     "powershell.exe", "\\?\\conhost.exe 0xffffffff -ForceV1",
                  "SYSTEM",      False, 0,  "WKSTN-101", 10), "Normal PS conhost"),
]

flagged = []
for event, desc in test_events:
    result = analyze_process_event(event)
    print(f"\n{desc}")
    print(f"  Process: {event.parent_name} -> {event.process_name} | Score: {result['score']:.3f} [{result['severity']}]")
    for f in result["findings"]:
        print(f"  [{f['type']}] {f['detail']} (score={f['score']:.2f})")
    if result["score"] >= 0.70:
        flagged.append((event, result, desc))

if flagged:
    print(f"\n[EDR] {len(flagged)} high-risk events — LLM triage...")
    for event, result, desc in flagged[:1]:
        analysis = llm_analyze(
            f"EDR alert: {event.parent_name} spawned {event.process_name} on {event.host}.\n"
            f"Command: {event.command_line[:120]}\n"
            f"Findings: {[f['detail'] for f in result['findings']]}\n"
            f"Score {result['score']:.3f}. MITRE ATT&CK chain? Containment action? Under 80 words.",
            max_tokens=120
        )
        print(f"\n  [LLM] {analysis}")

## Demo 3: Container Security Scanner

**Kubernetes and Docker misconfigurations** are the cloud-native equivalent of
security group `0.0.0.0/0` rules — they are everywhere, they are dangerous,
and most developers don't realize what they've done.

Static analysis tools like `checkov` or `kube-score` flag individual settings.
The difference with an **LLM-powered scanner**: it understands *combinations* —
a container running as root AND mounted with the host filesystem AND privileged
is not three medium findings; it's one critical container escape path.

This demo implements a Dockerfile and Kubernetes manifest checker with:
- Rule-based individual finding detection (25 checks)
- Severity compounding: dangerous combinations score higher than sum of parts
- LLM-generated fix: specific `securityContext` YAML, not just "don't use root"

| Risk Category | Examples |
|---------------|----------|
| **Container escape** | `privileged: true`, host PID/network namespace, host volume mounts |
| **Privilege escalation** | `allowPrivilegeEscalation: true`, running as root (UID 0) |
| **Credential exposure** | ENV secrets, unencrypted ConfigMaps with tokens |
| **Supply chain** | Unpinned base images (`:latest`), missing image digest pinning |

In [ ]:
# ── Demo 3: Container Security Scanner ────────────────────────────────────────

@dataclass
class ContainerFinding:
    """A single container security finding."""
    check_id: str
    severity: str           # CRITICAL / HIGH / MEDIUM / LOW
    base_score: float
    title: str
    description: str
    remediation: str        # Quick inline fix suggestion

def scan_dockerfile(dockerfile_lines: List[str]) -> List[ContainerFinding]:
    """Scan Dockerfile lines for security misconfigurations."""
    findings = []
    full_text = "\n".join(dockerfile_lines)

    # Check: FROM with :latest tag
    if re.search(r'^FROM\s+\S+:latest', full_text, re.MULTILINE | re.IGNORECASE):
        findings.append(ContainerFinding(
            "DF-001", "HIGH", 0.65,
            "Unpinned base image (:latest)",
            "Using ':latest' tag means the image can change silently — introducing vulnerabilities.",
            "Pin to a specific digest: FROM ubuntu@sha256:abc123..."
        ))

    # Check: running as root (no USER directive or USER root)
    user_directives = re.findall(r'^USER\s+(\S+)', full_text, re.MULTILINE | re.IGNORECASE)
    if not user_directives or any(u.lower() in ("root", "0") for u in user_directives):
        findings.append(ContainerFinding(
            "DF-002", "HIGH", 0.70,
            "Container runs as root",
            "If the container is compromised, the attacker has root in the pod and potentially on the node.",
            "Add: USER 1000  (or create a named non-root user with RUN useradd -u 1000 appuser)"
        ))

    # Check: secrets in ENV
    env_lines = [l for l in dockerfile_lines if l.strip().upper().startswith("ENV")]
    secret_keywords = ["password", "secret", "token", "key", "api_key", "passwd", "credential"]
    for line in env_lines:
        if any(kw in line.lower() for kw in secret_keywords):
            findings.append(ContainerFinding(
                "DF-003", "CRITICAL", 0.90,
                "Secret in Dockerfile ENV directive",
                f"ENV secrets are visible in 'docker inspect', image layers, and Kubernetes pod specs.",
                "Remove ENV secrets. Use Kubernetes Secrets + volumeMount or external secret manager (Vault)."
            ))
            break

    # Check: ADD instead of COPY (ADD can fetch remote URLs)
    if re.search(r'^ADD\s+http', full_text, re.MULTILINE | re.IGNORECASE):
        findings.append(ContainerFinding(
            "DF-004", "MEDIUM", 0.45,
            "ADD fetching remote URL",
            "ADD with a URL fetches arbitrary content at build time — supply chain risk.",
            "Use COPY for local files. Fetch remote content explicitly with verified checksums."
        ))

    # Check: EXPOSE 22 (SSH in container)
    if re.search(r'^EXPOSE\s+22\b', full_text, re.MULTILINE):
        findings.append(ContainerFinding(
            "DF-005", "HIGH", 0.75,
            "SSH exposed in container (EXPOSE 22)",
            "Containers should not run SSH. If shell access is needed, use kubectl exec.",
            "Remove EXPOSE 22 and the SSH server installation from the Dockerfile."
        ))

    return findings

def scan_k8s_manifest(manifest: dict) -> List[ContainerFinding]:
    """Scan a Kubernetes pod/deployment manifest dict for security misconfigurations."""
    findings = []
    containers = manifest.get("spec", {}).get("containers", [])

    for container in containers:
        name = container.get("name", "unknown")
        sc = container.get("securityContext", {})

        if sc.get("privileged", False):
            findings.append(ContainerFinding(
                f"K8S-001[{name}]", "CRITICAL", 0.95,
                f"Container '{name}' runs as privileged",
                "privileged: true gives the container near-root access to the host kernel. "
                "One container escape = full node compromise.",
                "Set: securityContext.privileged: false  (never use true in production)"
            ))

        if sc.get("allowPrivilegeEscalation", True):  # default is True if not set
            findings.append(ContainerFinding(
                f"K8S-002[{name}]", "HIGH", 0.70,
                f"Container '{name}': allowPrivilegeEscalation not disabled",
                "Allows setuid binaries to escalate privileges inside the container.",
                "Add: securityContext.allowPrivilegeEscalation: false"
            ))

        run_as_user = sc.get("runAsUser")
        run_as_non_root = sc.get("runAsNonRoot", False)
        if run_as_user == 0 or (run_as_user is None and not run_as_non_root):
            findings.append(ContainerFinding(
                f"K8S-003[{name}]", "HIGH", 0.72,
                f"Container '{name}': runs as root or runAsNonRoot not enforced",
                "Container process may run as root user (UID 0).",
                "Add: securityContext.runAsNonRoot: true  and  runAsUser: 1000"
            ))

        if not container.get("resources"):
            findings.append(ContainerFinding(
                f"K8S-004[{name}]", "MEDIUM", 0.40,
                f"Container '{name}': no resource limits defined",
                "Without limits, a compromised container can exhaust node CPU/memory (DoS).",
                "Add resources.limits.cpu and resources.limits.memory to the container spec."
            ))

        # Check for secret env vars
        for env in container.get("env", []):
            if any(kw in env.get("name", "").lower()
                   for kw in ["password", "secret", "token", "key", "api_key"]):
                if "valueFrom" not in env:  # hardcoded, not from Secret
                    findings.append(ContainerFinding(
                        f"K8S-005[{name}]", "CRITICAL", 0.88,
                        f"Container '{name}': secret in plain-text env var '{env['name']}'",
                        "Secret visible in 'kubectl describe pod' output and stored in etcd unencrypted.",
                        f"Replace with: env.valueFrom.secretKeyRef pointing to a Kubernetes Secret."
                    ))

    # Pod-level checks
    pod_sc = manifest.get("spec", {}).get("securityContext", {})
    if not pod_sc.get("seccompProfile"):
        findings.append(ContainerFinding(
            "K8S-006", "MEDIUM", 0.45,
            "No seccomp profile defined on pod",
            "Without seccomp, the container can make any system call — expanding attack surface.",
            "Add: securityContext.seccompProfile.type: RuntimeDefault"
        ))

    return findings

def compound_score(findings: List[ContainerFinding]) -> float:
    """Compound scoring: dangerous combinations score higher than sum of parts."""
    scores = [f.base_score for f in findings]
    if not scores:
        return 0.0
    max_score = max(scores)
    # Bonus for combinations: privileged + root + secret = worse than each alone
    critical_count = sum(1 for f in findings if f.severity == "CRITICAL")
    combo_bonus = min(critical_count * 0.05, 0.15)
    return min(max_score + combo_bonus, 1.0)

# ── Scan sample artifacts ──────────────────────────────────────────────────────
print("=" * 68)
print("CONTAINER SECURITY SCANNER — DOCKERFILE + K8S MANIFEST")
print("=" * 68)

# Vulnerable Dockerfile
bad_dockerfile = [
    "FROM ubuntu:latest",
    "RUN apt-get update && apt-get install -y openssh-server python3",
    "ENV DATABASE_PASSWORD=SuperSecret123",
    "ENV API_TOKEN=sk-live-abc123xyz",
    "ADD https://example.com/setup.sh /tmp/setup.sh",
    "RUN chmod +x /tmp/setup.sh && /tmp/setup.sh",
    "EXPOSE 22",
    "EXPOSE 8080",
    "CMD [\"python3\", \"/app/server.py\"]",
]

# Vulnerable K8s deployment
bad_k8s = {
    "kind": "Deployment",
    "metadata": {"name": "webapp"},
    "spec": {
        "containers": [{
            "name": "webapp",
            "image": "myapp:1.2",
            "securityContext": {
                "privileged": True,
                "runAsUser": 0,
                "allowPrivilegeEscalation": True,
            },
            "env": [
                {"name": "DB_PASSWORD", "value": "prod-secret-123"},
                {"name": "API_TOKEN",   "value": "bearer-xyz-456"},
            ],
            # no resources defined
        }]
    }
}

df_findings = scan_dockerfile(bad_dockerfile)
k8s_findings = scan_k8s_manifest(bad_k8s)
all_container_findings = df_findings + k8s_findings

print("\n-- Dockerfile findings --")
for f in df_findings:
    print(f"  [{f.check_id}] {f.severity} ({f.base_score:.2f}) — {f.title}")
    print(f"    Fix: {f.remediation}")

print("\n-- Kubernetes manifest findings --")
for f in k8s_findings:
    print(f"  [{f.check_id}] {f.severity} ({f.base_score:.2f}) — {f.title}")

overall = compound_score(all_container_findings)
print(f"\n[Container Scanner] Total findings: {len(all_container_findings)} | "
      f"Compounded risk score: {overall:.3f}")

# LLM generates specific remediated securityContext
critical_findings = [f for f in all_container_findings if f.severity == "CRITICAL"]
if critical_findings:
    issues = "; ".join(f.title for f in critical_findings[:3])
    analysis = llm_analyze(
        f"Container security: Kubernetes deployment has these critical issues: {issues}.\n"
        f"The container is privileged, runs as root, and has secrets in plain-text env vars.\n"
        f"Write the corrected securityContext YAML snippet and secret handling pattern. Under 120 words.",
        max_tokens=180
    )
    print(f"\n[LLM Remediation Plan]\n{analysis}")

## Demo 4: Cloud Workload Anomaly Detector

Even if your configuration is perfect, workload behavior can reveal compromise.
A production EC2 instance that suddenly spikes CPU to 98% and makes outbound
connections to a new IP range isn't misconfigured — it's behaving abnormally.

This demo detects **workload behavioral anomalies** using **statistical z-score
deviation** from a rolling baseline — no ML libraries, pure Python statistics:

```
z_score = (current_value - baseline_mean) / baseline_stddev

|z| > 2.0  = elevated (worth logging)
|z| > 3.0  = anomalous (alert)
|z| > 4.5  = critical  (auto-investigate)
```

Three metrics are monitored per workload:

| Metric | What It Catches |
|--------|----------------|
| **CPU utilization %** | Cryptomining (sustained 95%+), scanning, brute-force |
| **Outbound bytes/hour** | Data exfiltration spikes |
| **AWS API calls/hour** | IAM enumeration, lateral movement via cloud APIs |

Correlated anomalies across metrics produce a much higher confidence signal than
any single metric alone.

In [ ]:
# ── Demo 4: Cloud Workload Anomaly Detector ────────────────────────────────────

@dataclass
class WorkloadBaseline:
    """Rolling 30-day behavioral baseline for one cloud workload."""
    instance_id: str
    instance_type: str
    # CPU %
    cpu_mean: float
    cpu_std: float
    # Outbound bytes/hour
    net_out_mean: float
    net_out_std: float
    # AWS API calls/hour
    api_calls_mean: float
    api_calls_std: float

@dataclass
class WorkloadSnapshot:
    """Current metrics for a workload at one observation point."""
    instance_id: str
    cpu_pct: float
    net_out_bytes_per_hour: float
    api_calls_per_hour: int

def z_score(value: float, mean: float, std: float) -> float:
    """Compute z-score. Safe: returns 0 if std is 0."""
    if std <= 0:
        return 0.0
    return (value - mean) / std

def score_z(z: float) -> float:
    """Convert z-score to 0.0-1.0 risk score."""
    z = abs(z)
    if z < 2.0:  return 0.0
    if z < 3.0:  return 0.30
    if z < 4.5:  return 0.60
    return min(0.60 + (z - 4.5) * 0.08, 1.0)

def detect_workload_anomaly(baseline: WorkloadBaseline, snapshot: WorkloadSnapshot) -> dict:
    """
    Compare current workload metrics against 30-day baseline using z-scores.
    Returns anomaly findings and combined risk score.
    """
    metrics = {
        "cpu_pct": (
            z_score(snapshot.cpu_pct, baseline.cpu_mean, baseline.cpu_std),
            snapshot.cpu_pct, baseline.cpu_mean, "%"
        ),
        "net_out_bytes_hr": (
            z_score(snapshot.net_out_bytes_per_hour, baseline.net_out_mean, baseline.net_out_std),
            snapshot.net_out_bytes_per_hour, baseline.net_out_mean, " B/hr"
        ),
        "api_calls_hr": (
            z_score(snapshot.api_calls_per_hour, baseline.api_calls_mean, baseline.api_calls_std),
            snapshot.api_calls_per_hour, baseline.api_calls_mean, " calls/hr"
        ),
    }

    findings = []
    scores = []
    for metric_name, (z, current, mean, unit) in metrics.items():
        s = score_z(z)
        if s > 0:
            severity = "CRITICAL" if s >= 0.75 else ("HIGH" if s >= 0.60 else "MEDIUM")
            findings.append({
                "metric": metric_name,
                "z_score": round(z, 2),
                "current": current,
                "baseline_mean": mean,
                "risk_score": round(s, 3),
                "severity": severity,
                "unit": unit,
            })
            scores.append(s)

    # Correlation bonus: multiple anomalous metrics simultaneously = higher confidence
    anomalous_metrics = len([s for s in scores if s >= 0.30])
    base_score = max(scores) if scores else 0.0
    correlation_bonus = min(anomalous_metrics * 0.05, 0.15)
    final_score = min(base_score + correlation_bonus, 1.0)

    return {
        "instance_id": snapshot.instance_id,
        "final_score": round(final_score, 3),
        "anomalous_metrics": anomalous_metrics,
        "findings": findings,
    }

# ── Workload baselines (30-day rolling averages) ───────────────────────────────
baselines = {
    "i-0abc123prod": WorkloadBaseline(
        "i-0abc123prod", "t3.large",
        cpu_mean=22.0,      cpu_std=8.0,
        net_out_mean=5e6,   net_out_std=2e6,      # ~5 MB/hr normal
        api_calls_mean=120, api_calls_std=30,
    ),
    "i-0def456dev": WorkloadBaseline(
        "i-0def456dev", "t3.small",
        cpu_mean=10.0,      cpu_std=5.0,
        net_out_mean=1e6,   net_out_std=5e5,
        api_calls_mean=50,  api_calls_std=20,
    ),
}

# ── Current snapshots ─────────────────────────────────────────────────────────
print("=" * 68)
print("CLOUD WORKLOAD ANOMALY DETECTOR — REAL-TIME METRIC ANALYSIS")
print("=" * 68)

snapshots = [
    WorkloadSnapshot("i-0abc123prod", cpu_pct=21.0,  net_out_bytes_per_hour=5.2e6, api_calls_per_hour=115),  # normal
    WorkloadSnapshot("i-0abc123prod", cpu_pct=97.5,  net_out_bytes_per_hour=4.8e6, api_calls_per_hour=118),  # cryptomining?
    WorkloadSnapshot("i-0abc123prod", cpu_pct=24.0,  net_out_bytes_per_hour=820e6, api_calls_per_hour=890),  # exfil + API enum
    WorkloadSnapshot("i-0def456dev",  cpu_pct=9.0,   net_out_bytes_per_hour=1.1e6, api_calls_per_hour=48),   # normal
    WorkloadSnapshot("i-0def456dev",  cpu_pct=93.0,  net_out_bytes_per_hour=310e6, api_calls_per_hour=1440), # fully compromised
]

descriptions = [
    "prod: normal weekday afternoon",
    "prod: CPU spike — cryptominer suspected",
    "prod: exfil + IAM enumeration",
    "dev: normal idle",
    "dev: full compromise — cryptominer + exfil + API",
]

high_risk_workloads = []
for snapshot, desc in zip(snapshots, descriptions):
    baseline = baselines[snapshot.instance_id]
    result = detect_workload_anomaly(baseline, snapshot)

    sev_tag = ""
    if result["final_score"] >= 0.70:
        sev_tag = " [CRITICAL]"
        high_risk_workloads.append((snapshot, result, desc))
    elif result["final_score"] >= 0.40:
        sev_tag = " [HIGH]"

    print(f"\n{desc}")
    print(f"  {snapshot.instance_id}: score={result['final_score']:.3f} "
          f"anomalous_metrics={result['anomalous_metrics']}{sev_tag}")
    for f in result["findings"]:
        current_str = f"{f['current']:.1e}" if f['current'] > 1000 else f"{f['current']:.1f}"
        mean_str = f"{f['baseline_mean']:.1e}" if f['baseline_mean'] > 1000 else f"{f['baseline_mean']:.1f}"
        print(f"  [{f['severity']}] {f['metric']}: z={f['z_score']:.1f} "
              f"current={current_str}{f['unit']} baseline={mean_str}{f['unit']}")

# LLM analysis for critical workloads
if high_risk_workloads:
    snapshot, result, desc = high_risk_workloads[-1]  # most interesting case
    analysis = llm_analyze(
        f"Cloud workload anomaly: {snapshot.instance_id}\n"
        f"CPU: {snapshot.cpu_pct}% (baseline ~10%), "
        f"Net out: {snapshot.net_out_bytes_per_hour:.0e} B/hr (baseline ~1M), "
        f"API calls: {snapshot.api_calls_per_hour}/hr (baseline ~50).\n"
        f"All three metrics anomalous simultaneously. Threat classification? "
        f"Immediate containment action? MITRE cloud technique? Under 80 words.",
        max_tokens=120
    )
    print(f"\n[LLM Threat Classification] {analysis}")

## Demo 5: Full Cloud Security Posture Pipeline

The full **CSPM pipeline** wires all four engines into an end-to-end
cloud security assessment — the way a mature security organization
runs continuous posture management:

```
Config snapshots  ->  Drift Detector       -+
Process events    ->  Endpoint Baseline     |
Container specs   ->  Container Scanner     +-> Risk Prioritizer -> LLM Remediation -> Report
Workload metrics  ->  Anomaly Detector     -+
```

The **risk prioritizer** uses a weighted combination of finding severity,
resource exposure (internet-facing?), and data sensitivity to generate
a prioritized remediation queue — not 10,000 unordered findings, but
a top-10 list with specific fix instructions.

**Key insight**: a public-facing EC2 with a privileged container AND
a misconfigured IAM role AND anomalous CPU is not three separate findings —
it's one critical attack path requiring emergency response.

> **Non-negotiable guardrail: All remediation actions require human approval.
> The pipeline generates the plan. The engineer approves and executes.**

In [ ]:
# ── Demo 5: Full Cloud Security Posture Pipeline ──────────────────────────────

@dataclass
class SecurityFinding:
    """Normalized finding from any scanner, ready for risk prioritization."""
    finding_id: str
    source: str           # DRIFT / ENDPOINT / CONTAINER / WORKLOAD
    severity: str
    risk_score: float     # 0.0-1.0
    resource_id: str
    description: str
    remediation: str
    internet_facing: bool = False
    contains_sensitive_data: bool = False

class CloudSecurityPipeline:
    """
    Full CSPM pipeline: ingest findings from all engines,
    apply contextual risk adjustment, prioritize, generate remediation plan.
    """

    def __init__(self):
        self.all_findings: List[SecurityFinding] = []
        self._counter = 0

    def _next_id(self, source: str) -> str:
        self._counter += 1
        return f"{source}-{self._counter:04d}"

    def add_drift_findings(self, drift_findings: List[DriftFinding],
                           resource_context: dict):
        """Convert drift findings to normalized SecurityFindings with context."""
        for df in drift_findings:
            self.all_findings.append(SecurityFinding(
                finding_id=self._next_id("DRIFT"),
                source="CLOUD_DRIFT",
                severity=df.severity,
                risk_score=df.risk_score,
                resource_id=df.resource_id,
                description=df.description,
                remediation=f"Revert {df.field} from '{df.current_value}' to '{df.baseline_value}'",
                internet_facing=resource_context.get(df.resource_id, {}).get("internet_facing", False),
                contains_sensitive_data=resource_context.get(df.resource_id, {}).get("sensitive", False),
            ))

    def add_container_findings(self, container_findings: List[ContainerFinding],
                               internet_facing: bool = False):
        """Convert container findings."""
        for cf in container_findings:
            self.all_findings.append(SecurityFinding(
                finding_id=self._next_id("CONT"),
                source="CONTAINER",
                severity=cf.severity,
                risk_score=cf.base_score,
                resource_id=cf.check_id,
                description=cf.description,
                remediation=cf.remediation,
                internet_facing=internet_facing,
            ))

    def add_workload_finding(self, workload_result: dict, internet_facing: bool = True,
                              sensitive_data: bool = False):
        """Convert workload anomaly to SecurityFinding."""
        if workload_result["final_score"] < 0.40:
            return
        metrics_summary = "; ".join(
            f"{f['metric']}(z={f['z_score']:.1f})"
            for f in workload_result.get("findings", [])
        )
        self.all_findings.append(SecurityFinding(
            finding_id=self._next_id("WKLD"),
            source="WORKLOAD",
            severity="CRITICAL" if workload_result["final_score"] >= 0.75 else "HIGH",
            risk_score=workload_result["final_score"],
            resource_id=workload_result["instance_id"],
            description=f"Workload anomaly: {metrics_summary}",
            remediation="Isolate instance, capture forensic snapshot, analyze memory and process tree",
            internet_facing=internet_facing,
            contains_sensitive_data=sensitive_data,
        ))

    def prioritize(self) -> List[SecurityFinding]:
        """
        Apply contextual risk multipliers and sort by adjusted score.
        Internet-facing + sensitive data = higher effective priority.
        """
        def adjusted_score(f: SecurityFinding) -> float:
            score = f.risk_score
            if f.internet_facing:           score = min(score * 1.25, 1.0)
            if f.contains_sensitive_data:   score = min(score * 1.15, 1.0)
            return score

        return sorted(self.all_findings, key=adjusted_score, reverse=True)

    def generate_llm_remediation(self, top_findings: List[SecurityFinding]) -> str:
        """Ask LLM for a prioritized remediation plan for the top critical findings."""
        summary = "\n".join(
            f"- [{f.source}] {f.severity}: {f.description} (resource: {f.resource_id})"
            for f in top_findings[:5]
        )
        return llm_analyze(
            f"Cloud security assessment for production AWS environment.\n"
            f"Top priority findings requiring immediate action:\n{summary}\n\n"
            f"Provide a numbered remediation plan: order by blast radius, "
            f"include who owns each action (Security / DevOps / Platform). Under 150 words.",
            max_tokens=220
        )

# ── Run the full pipeline ──────────────────────────────────────────────────────
print("=" * 70)
print("FULL CLOUD SECURITY POSTURE PIPELINE")
print("=" * 70)

pipeline = CloudSecurityPipeline()

# Resource context: which resources are internet-facing or hold sensitive data
resource_context = {
    "prod-lambda-execution-role": {"internet_facing": True,  "sensitive": True},
    "sg-prod-rds-db-01":          {"internet_facing": True,  "sensitive": True},
    "prod-customer-data-bucket":  {"internet_facing": False, "sensitive": True},
}

# Ingest drift findings (from Demo 1)
drift_findings_from_demo1 = []
drift_findings_from_demo1 += score_iam_drift(
    "prod-lambda-execution-role",
    baseline={"allowed_actions": ["s3:GetObject", "logs:CreateLogGroup"],
              "resource_scope": ["arn:aws:s3:::prod-data-bucket/*"], "require_mfa": True},
    current={"allowed_actions": ["s3:GetObject", "logs:CreateLogGroup", "s3:*", "iam:PassRole"],
             "resource_scope": ["*"], "require_mfa": False}
)
drift_findings_from_demo1 += score_sg_drift(
    "sg-prod-rds-db-01",
    baseline={"ingress_rules": [{"port": 5432, "cidr": "10.0.1.0/24"}]},
    current={"ingress_rules": [{"port": 5432, "cidr": "0.0.0.0/0"}, {"port": 22, "cidr": "0.0.0.0/0"}]}
)
drift_findings_from_demo1 += score_s3_drift(
    "prod-customer-data-bucket",
    baseline={"public_access_blocked": True, "encryption_enabled": True,
              "logging_enabled": True, "cross_account_principals": []},
    current={"public_access_blocked": False, "encryption_enabled": True,
             "logging_enabled": False, "cross_account_principals": ["arn:aws:iam::999888777666:root"]}
)
pipeline.add_drift_findings(drift_findings_from_demo1, resource_context)

# Ingest container findings (from Demo 3)
k8s_findings_2 = scan_k8s_manifest(bad_k8s)
pipeline.add_container_findings(k8s_findings_2, internet_facing=True)

# Ingest workload anomaly (from Demo 4)
workload_baseline = baselines["i-0def456dev"]
critical_snapshot = WorkloadSnapshot("i-0def456dev", cpu_pct=93.0,
                                      net_out_bytes_per_hour=310e6, api_calls_per_hour=1440)
wl_result = detect_workload_anomaly(workload_baseline, critical_snapshot)
pipeline.add_workload_finding(wl_result, internet_facing=True, sensitive_data=True)

# Prioritize
prioritized = pipeline.prioritize()

print(f"\n[Pipeline] Total findings ingested: {len(prioritized)}")
print(f"  CRITICAL: {sum(1 for f in prioritized if f.severity=='CRITICAL')}")
print(f"  HIGH:     {sum(1 for f in prioritized if f.severity=='HIGH')}")
print(f"  MEDIUM:   {sum(1 for f in prioritized if f.severity=='MEDIUM')}")

print("\n-- PRIORITIZED REMEDIATION QUEUE (Top 8) --")
print(f"{'#':<3} {'ID':<12} {'Source':<14} {'Sev':<10} {'Score':>6} {'Internet':>8} {'Sensitive':>9}")
print("-" * 72)
for i, f in enumerate(prioritized[:8], 1):
    net = "YES" if f.internet_facing else "no"
    sens = "YES" if f.contains_sensitive_data else "no"
    print(f"{i:<3} {f.finding_id:<12} {f.source:<14} {f.severity:<10} {f.risk_score:>6.3f} {net:>8} {sens:>9}")
    print(f"    {f.description[:80]}")

# LLM generates the remediation plan
print("\n" + "=" * 70)
print("LLM REMEDIATION PLAN (awaiting security team approval)")
print("=" * 70)
top_critical = [f for f in prioritized if f.severity == "CRITICAL"]
if top_critical:
    plan = pipeline.generate_llm_remediation(top_critical)
    print(plan)

print("\n" + "=" * 70)
print("GUARDRAIL: All actions above require explicit security team approval.")
print("No automated remediation. The pipeline plans — engineers execute.")

## Summary: What You Built

You now have a complete **AI-Powered Cloud & Endpoint Security** system with 5 engines:

| Engine | Technique | Key Metric | Threshold |
|--------|-----------|------------|----------|
| **Cloud Drift Detector** | Baseline diff + rule scoring | Per-resource risk score | > 0.70 = CRITICAL |
| **Endpoint Behavioral Baseline** | Attack chain + cmdline pattern + z-score | Process anomaly score | > 0.70 = alert |
| **Container Scanner** | Dockerfile + K8s manifest checks + compound scoring | Compounded risk | > 0.80 = critical |
| **Workload Anomaly Detector** | Z-score deviation on CPU/net/API + correlation bonus | Z-score | |z| > 3 = anomaly |
| **Full CSPM Pipeline** | Multi-source ingestion + context-adjusted prioritization | Adjusted score | Top-N remediation queue |

### Why Rule-Based CSPM Isn't Enough

- **Rules**: catch configurations the rule writer anticipated — miss novel combinations and unknown drifts
- **AI-enhanced CSPM**: detects behavioral anomalies and relationship-aware risk — catches what no individual rule would

### The Non-Negotiable Rule

> **All remediation actions require human engineer approval before execution.**
> The AI pipeline generates the plan and the priority order.
> The security team reviews and executes. No automated production changes.

### Production Upgrade Path

```
Dict-based baseline storage    ->  CloudWatch / Prometheus time-series database
Manual snapshot comparison     ->  AWS Config Rules + continuous drift detection
In-memory finding queue        ->  SIEM integration (Splunk / Elastic / Sentinel)
Simulated LLM remediation      ->  claude-sonnet-4-5-20250514 for complex analysis
Per-resource risk scoring      ->  Graph DB (JupiterOne / Neptune) for relationship-aware scoring
Static container rule checks   ->  Integration with Trivy / Checkov in CI/CD pipeline
```

**Next: Chapter 90 — AI-Augmented SOC Operations** — connecting all the security
AI capabilities from Volume 4 into a mature, unified Security Operations Center
with coherent detection, investigation, and response workflows.

In [ ]:
# ── Chapter 89: Production Deployment Checklist ────────────────────────────────
print("CHAPTER 89: CLOUD & ENDPOINT SECURITY — PRODUCTION CHECKLIST")
print("=" * 65)

checklist = [
    ("Cloud Config Data",    "AWS Config / Azure Policy -> continuous drift detection"),
    ("Cloud Config Data",    "Terraform state -> compare deployed vs declared"),
    ("Cloud Config Data",    "CloudTrail API logs -> IAM enumeration detection"),
    ("Cloud Config Data",    "VPC Flow Logs -> workload network anomaly baseline"),
    ("Endpoint Data",        "EDR telemetry: process events, API calls, network connections"),
    ("Endpoint Data",        "Windows Event Log 4688 (process creation) -> baseline"),
    ("Endpoint Data",        "Sysmon EventID 1 + 3 + 7 for process + network + image load"),
    ("Container Security",   "Scan Dockerfiles in CI/CD — fail build on CRITICAL findings"),
    ("Container Security",   "Admission controller (OPA/Kyverno) to enforce pod security"),
    ("Container Security",   "Image registry scanning (Trivy) — block on known CVEs"),
    ("Workload Baseline",    "30-day rolling z-score baseline per instance type + role"),
    ("Workload Baseline",    "Separate baselines: business hours vs nights/weekends"),
    ("Workload Baseline",    "Correlation window: 15 minutes for multi-metric anomalies"),
    ("LLM Integration",      "Model: claude-haiku-4-5-20251001 for triage and classification"),
    ("LLM Integration",      "Model: claude-sonnet-4-5-20250514 for remediation plan generation"),
    ("LLM Integration",      "RAG: CIS Benchmarks + NIST CSF + internal runbooks"),
    ("Guardrails",           "HUMAN APPROVAL before any production remediation"),
    ("Guardrails",           "Immutable audit log: every finding, every action, every approval"),
    ("Guardrails",           "Rollback tested: can you revert each remediation within 5 minutes?"),
    ("Guardrails",           "False positive budget: track and tune — aim for <5% FP rate"),
]

current = ""
for category, item in checklist:
    if category != current:
        print(f"\n[{category}]")
        current = category
    print(f"  + {item}")

print("\n" + "=" * 65)
print("KEY FORMULAS")
print("=" * 65)
print("Drift risk score  = max(per-field scores) + compound bonus for co-occurring CRITICAL")
print("Endpoint score    = max(attack_chain, cmdline_pattern, child_count, system_netconn)")
print("Container score   = max(per-check scores) + 0.05 per critical finding (capped +0.15)")
print("Workload z-score  = (current - mean) / stddev  |  |z|>3.0 = alert  |  |z|>4.5 = critical")
print("CSPM priority     = risk_score * 1.25 (internet-facing) * 1.15 (sensitive data)")
print("\nAll pipelines merge into unified finding queue -> LLM plan -> human approval -> execute")
print("\nChapter 89 complete. The attack surface is too large to see without AI. Now you can see it.")